# EvoCRM v2: Module Validation Notebook

Per-module smoke tests. Run top-to-bottom to validate each component.

**Sections:**
1. Data generation
2. Labels (leak-free)
3. Features (cutoff-gated)
4. FT-Transformer (Customer Tower)
5. Perceiver IO hub
6. Full EvoCRM model forward pass
7. Training loop (mini)
8. Evaluation
9. LoRA injection and parameter counting
10. Pretraining + transfer (multi-dataset)
11. Baselines comparison

In [ ]:
import sys, os
sys.path.insert(0, '.')
import numpy as np
import pandas as pd
import torch
np.random.seed(0)
torch.manual_seed(0)
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 1. Data generation

In [ ]:
from evocrm.data import generate_synthetic_olist, EVOCRM_SCHEMA

tables = generate_synthetic_olist(n_customers=500, n_products=80, seed=1)
for name, df in tables.items():
    assert set(EVOCRM_SCHEMA[name]).issubset(df.columns), f'{name} schema violation'
    print(f'{name:>14s}: {df.shape}')

## 2. Labels (leak-free)

Invariants:
- Every row with `churn=1` has `clv=0` (by construction).
- `category_valid` is False for churners (no next order).
- Churn rate is non-trivial (5–50%). Extremes indicate data issues.

In [ ]:
from evocrm.labels import LabelingConfig, build_all_labels, suggest_cutoff

cutoff = suggest_cutoff(tables['orders'], observation_days=90)
cfg = LabelingConfig(cutoff_date=cutoff, observation_window_days=90)
labels = build_all_labels(
    tables['orders'], tables['order_items'],
    tables['products'], tables['customers'], cfg,
)

assert labels.loc[labels['churn']==1, 'clv'].max() == 0, 'LEAK: churners have non-zero CLV'
assert (labels.loc[labels['churn']==1, 'category_valid']==False).all(), 'churners should have invalid category'
print(f"churn rate: {labels['churn'].mean():.3f}  (0.05-0.50 expected)")
print(f'n labeled:  {len(labels)}')
labels.head()

## 3. Features (cutoff-gated)

Invariant: `recency_days >= 0` for every customer (any negative value = post-cutoff leak).

In [ ]:
from evocrm.features import (
    build_customer_features, build_interaction_sequences,
    assemble_feature_matrix,
)

cust_feats = build_customer_features(
    tables['orders'], tables['order_items'],
    tables['customers'], cutoff,
)
assert (cust_feats['recency_days'] >= 0).all(), 'NEGATIVE RECENCY = feature-side leak'
print(f'feature columns: {list(cust_feats.columns)}')
cust_feats.describe()

In [ ]:
seq_data = build_interaction_sequences(
    tables['orders'], tables['order_items'],
    tables['customers'], cutoff, max_length=20,
)
print(f'n customers with sequences: {len(seq_data.customer_ids)}')
print(f'product_ids shape:          {seq_data.product_ids.shape}')
print(f'mask density (avg non-pad): {seq_data.mask.mean():.3f}')
print(f'product vocab size:         {seq_data.product_id_vocab_size}')

## 4. FT-Transformer (Customer Tower)

Check: input `(B, F)` → output `(B, F+1, d_token)` with CLS at index 0.

In [ ]:
from evocrm.ft_transformer import FTTransformer

ft = FTTransformer(n_features=6, d_token=32, n_blocks=2, n_heads=4)
x = torch.randn(8, 6)
out = ft(x)
assert out.shape == (8, 7, 32), f'unexpected shape {out.shape}'
print(f'ok: (8,6) -> {tuple(out.shape)}')
print(f'params: {sum(p.numel() for p in ft.parameters()):,}')

## 5. Perceiver IO hub

Check: inputs `(B, N, d_in)` + mask `(B, N)` → latents `(B, N_latent, d_latent)`.

In [ ]:
from evocrm.perceiver import PerceiverIOHub, OutputQueryHead

hub = PerceiverIOHub(
    num_latents=16, latent_dim=64, input_dim=64,
    n_cross_attention_heads=4, n_self_attention_heads=4,
    n_self_attention_blocks_per_layer=1, n_layers=2,
)
inputs = torch.randn(4, 30, 64)
mask = torch.ones(4, 30, dtype=torch.bool)
mask[:, 20:] = False  # last 10 tokens are padding
latents = hub(inputs, input_mask=mask)
assert latents.shape == (4, 16, 64), f'unexpected {latents.shape}'
print(f'hub: {tuple(inputs.shape)} -> {tuple(latents.shape)}')

head = OutputQueryHead(latent_dim=64, query_dim=64, output_dim=1)
pred = head(latents)
print(f'head output: {tuple(pred.shape)}  (expected (4,1))')

## 6. Full EvoCRM model forward pass

In [ ]:
from evocrm.config import EvoCRMConfig
from evocrm.model import EvoCRM

cfg = EvoCRMConfig()
cfg.perceiver.num_latents = 16
cfg.perceiver.n_layers = 2
cfg.ft_transformer.n_blocks = 2

model = EvoCRM(cfg, n_tabular_features=6, product_vocab_size=100, n_categories=12)

B = 4
out = model(
    tabular=torch.randn(B, 6),
    seq_pids=torch.randint(0, 100, (B, 20)),
    seq_dt=torch.rand(B, 20) * 300,
    seq_mask=torch.ones(B, 20, dtype=torch.bool),
    cat_ids=torch.randint(0, 12, (B, 3)),
)
for name, pred in out.items():
    print(f'{name:>10s}: {tuple(pred.shape)}')

print(f'\ntotal params: {sum(p.numel() for p in model.parameters()):,}')

## 7. Training loop (mini, 3 epochs)

In [ ]:
from evocrm.pipeline import run_pipeline

small_cfg = EvoCRMConfig()
small_cfg.training.num_epochs = 3
small_cfg.training.batch_size = 32
small_cfg.training.device = 'cpu'
small_cfg.perceiver.n_layers = 2
small_cfg.ft_transformer.n_blocks = 2

artifacts = run_pipeline(tables, small_cfg, verbose=True)

## 8. Evaluation

In [ ]:
from evocrm.evaluate import evaluate
val = evaluate(artifacts.model, artifacts.val_dataset, device='cpu')
for task, metrics in val.items():
    print(f'{task}:')
    for k, v in metrics.items():
        print(f'   {k}: {v}')

## 9. LoRA injection and parameter counting

Inject LoRA, freeze base, verify trainable params drop dramatically.

In [ ]:
from evocrm.lora import (
    inject_lora_into_attention, freeze_non_lora, count_parameters,
)
import copy

m2 = copy.deepcopy(artifacts.model)
before = count_parameters(m2)
print(f'before LoRA: {before}')

n_wrapped = inject_lora_into_attention(m2, rank=8, alpha=16)
print(f'wrapped {n_wrapped} attention projections')

freeze_non_lora(m2)
after = count_parameters(m2)
print(f'after LoRA+freeze: {after}')
print(f'parameter reduction: {100 - after["trainable_pct"]:.1f}%')

## 10. Pretraining + transfer (multi-dataset)

Two synthetic source datasets → pretrain. Third → transfer.

In [ ]:
from evocrm.pretrain_transfer import transfer_evaluate

src_a = generate_synthetic_olist(n_customers=400, seed=11)
src_b = generate_synthetic_olist(n_customers=400, seed=22)
tgt = generate_synthetic_olist(n_customers=300, seed=33)

transfer_cfg = EvoCRMConfig()
transfer_cfg.training.num_epochs = 2
transfer_cfg.training.device = 'cpu'
transfer_cfg.perceiver.n_layers = 2
transfer_cfg.ft_transformer.n_blocks = 2

res = transfer_evaluate(
    source_tables={'src_a': src_a, 'src_b': src_b},
    target_tables=tgt,
    target_name='tgt',
    config=transfer_cfg,
    verbose=True,
)
print('\n\n=== TRANSFER COMPARISON ===')
print(f"LoRA trainable %: {res.lora_param_pct:.2f}")
print(f"baseline churn AUC:       {res.baseline_metrics['churn'].get('auc','na')}")
print(f"LoRA transfer churn AUC:  {res.lora_transfer_metrics['churn'].get('auc','na')}")
print(f"full finetune churn AUC:  {res.full_finetune_metrics['churn'].get('auc','na')}")

## 11. Baselines comparison

Specialist classical models for each task, apples-to-apples on the same tabular features.

In [ ]:
from evocrm.baselines import baselines_from_pipeline

bl = baselines_from_pipeline(artifacts, labels)
for task, methods in bl.items():
    print(f'\n{task}:')
    for method, m in methods.items():
        print(f'  {method}: {m}')

## Diagnostic: is churn AUC suspiciously high?

Rule of thumb on synthetic data: any AUC > 0.95 indicates a leak. 0.70–0.85 is the healthy range.

In [ ]:
auc = val['churn'].get('auc', 0.0)
if auc > 0.95:
    print(f'WARNING: churn AUC={auc:.3f} is suspiciously high. Check for leakage.')
elif auc < 0.55:
    print(f'NOTE: churn AUC={auc:.3f} is near chance. More epochs or bigger data needed.')
else:
    print(f'OK: churn AUC={auc:.3f} is in the healthy range (0.55-0.95).')